# Lot A — Exploration et analyse du dataset Online Retail

1. Qualité des données brutes (ce qu'il faut nettoyer)
2. Bilan du nettoyage
3. Analyse exploratoire : pays, produits, saisonnalité, heures de pointe

Les figures sont enregistrées dans `docs/captures/` pour le rapport.

> Lancer d'abord `python -m scripts.download_data` puis `python -m producer.prepare_data`.

In [ ]:
import sys, json
from pathlib import Path
import os
if Path.cwd().name == "notebooks":   # on se place à la racine du projet
    os.chdir(Path.cwd().parent)
sys.path.append(str(Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
from config import settings

FIG = Path("docs/captures"); FIG.mkdir(parents=True, exist_ok=True)
VIOLET = "#6C4CFF"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

## 1. Données brutes

In [ ]:
brut = pd.read_excel(settings.RAW_DATA, dtype={"InvoiceNo": str, "StockCode": str})
print(brut.shape)
brut.head()

In [ ]:
qualite = pd.DataFrame({
    "valeurs manquantes": brut.isna().sum(),
    "% manquant": (brut.isna().mean() * 100).round(1),
})
qualite

In [ ]:
annul = brut["InvoiceNo"].astype(str).str.startswith("C")
print("Doublons exacts            :", brut.duplicated().sum())
print("Factures annulées (lignes) :", annul.sum())
print("Quantités négatives        :", (brut["Quantity"] < 0).sum())
print("Prix <= 0                  :", (brut["UnitPrice"] <= 0).sum())
print("CustomerID manquant        :", brut["CustomerID"].isna().sum())
print("Période                    :", brut["InvoiceDate"].min(), "→", brut["InvoiceDate"].max())

In [ ]:
# Codes qui ne sont pas des produits (frais de port, ajustements…)
codes = brut["StockCode"].astype(str).str.upper()
brut[~codes.str.match(r"^\d{5}")].groupby(codes)["Description"].first().head(20)

## 2. Bilan du nettoyage

In [ ]:
stats = json.loads(Path("docs/nettoyage_stats.json").read_text(encoding="utf-8"))
pd.Series(stats, name="valeur").to_frame()

In [ ]:
df = pd.read_csv(settings.CLEAN_DATA, dtype={"InvoiceNo": str, "StockCode": str, "CustomerID": str},
                 parse_dates=["InvoiceDate"])
df["Montant"] = df["Quantity"] * df["UnitPrice"]
df["Annulation"] = df["InvoiceNo"].str.startswith("C")
ventes = df[~df["Annulation"]]
print(f"CA total : {ventes['Montant'].sum():,.0f} £  |  {ventes['InvoiceNo'].nunique():,} factures  |  "
      f"panier moyen : {ventes.groupby('InvoiceNo')['Montant'].sum().mean():.2f} £  |  "
      f"taux d'annulation : {df.loc[df['Annulation'], 'InvoiceNo'].nunique() / df['InvoiceNo'].nunique():.1%}")

## 3. Analyse exploratoire
### Chiffre d'affaires par pays

In [ ]:
pays = ventes.groupby("Country")["Montant"].sum().sort_values(ascending=False)
print(f"Part du Royaume-Uni : {pays.iloc[0] / pays.sum():.1%}")
ax = pays.head(10).iloc[::-1].plot.barh(color=VIOLET)
ax.set(title="Top 10 des pays par chiffre d'affaires", xlabel="CA (£)", ylabel="")
plt.tight_layout(); plt.savefig(FIG / "ca_par_pays.png", dpi=150); plt.show()

### Produits les plus vendus

In [ ]:
top = (ventes.groupby("Description")
       .agg(quantite=("Quantity", "sum"), ca=("Montant", "sum"))
       .sort_values("quantite", ascending=False).head(10))
ax = top["quantite"].iloc[::-1].plot.barh(color=VIOLET)
ax.set(title="Top 10 des produits (quantité vendue)", xlabel="Quantité", ylabel="")
plt.tight_layout(); plt.savefig(FIG / "top_produits.png", dpi=150); plt.show()
top

### Saisonnalité mensuelle
⚠️ Décembre 2011 est incomplet (le dataset s'arrête le 9).

In [ ]:
mois = ventes.set_index("InvoiceDate")["Montant"].resample("MS").sum()
ax = mois.plot(marker="o", color=VIOLET)
ax.set(title="Chiffre d'affaires par mois", xlabel="", ylabel="CA (£)")
plt.tight_layout(); plt.savefig(FIG / "ca_par_mois.png", dpi=150); plt.show()

### Jours et heures de pointe

In [ ]:
jours = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
par_jour = ventes.groupby(ventes["InvoiceDate"].dt.dayofweek)["Montant"].sum().reindex(range(7), fill_value=0)
par_heure = ventes.groupby(ventes["InvoiceDate"].dt.hour)["Montant"].sum()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.bar(jours, par_jour.values, color=VIOLET); a1.set(title="CA par jour de la semaine"); a1.tick_params(axis="x", rotation=45)
a2.bar(par_heure.index, par_heure.values, color=VIOLET); a2.set(title="CA par heure", xlabel="Heure")
plt.tight_layout(); plt.savefig(FIG / "jours_heures.png", dpi=150); plt.show()

### Volume de transactions par minute (dimensionnement du flux)
Utile pour choisir la vitesse de rejeu et la taille des fenêtres Spark.

In [ ]:
par_min = df.set_index("InvoiceDate").resample("1min").size()
par_min = par_min[par_min > 0]
par_min.describe().round(1)

## Conclusions à reporter dans le rapport
- *(à compléter avec les vrais chiffres)* part du Royaume-Uni, pays suivants
- produits phares
- pic de ventes avant Noël (sept.–nov. 2011)
- jours et heures de pointe (pas de ventes le samedi dans ce dataset)